In [1]:
import os
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (15.0, 12.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading extenrnal modules
%load_ext autoreload
%autoreload 2

torch.set_printoptions(sci_mode=False) # disable scientific notation
device = torch.device("cuda:0")

from dev_refraction.refraction_transform import *

In [2]:
# Setting Refraction Parameters
n = 1.33
plane = 0.0
num_newton_iters = 10
newton_atol = 1e-6

# Settings Gaussian and Camera Parameters
camera_height = 10
x = 10.0
y = 10.0
z = -10.0

means=torch.Tensor([[x, y, z]]).to(device)
cam_center=torch.Tensor([0.0, 0.0, camera_height]).to(device)

RT = RefractionTransform(
    means=means,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)

trandformed_means = RT.transform_to_appearance()
dra_dr = RT.dra_dr()
dra_dz = RT.dra_dz()
dza_dr = RT.dza_dr()
dza_dz = RT.dza_dz()
dPa_dP = RT.dPa_dP()



In [3]:
# Output inner values
print("RT inner values:")
for key, value in vars(RT).items():
    print(f"{key}: {value}")

RT inner values:
device: cuda:0
n: 1.3300000429153442
plane: 0.0
delta: 0.01
num_newton_iters: 10
newton_atol: 1e-06
means: tensor([[ 10.,  10., -10.]], device='cuda:0')
quats: None
num_g: 1
cam_center: tensor([ 0.,  0., 10.], device='cuda:0')
x0: 0.0
y0: 0.0
H: 10.0
x: tensor([10.], device='cuda:0')
y: tensor([10.], device='cuda:0')
z: tensor([-10.], device='cuda:0')
r: tensor([14.1421], device='cuda:0')
phi: tensor([0.7854], device='cuda:0')
n2: 1.7689001560211182
n2m1: 0.7689001560211182
H2: 100.0
r2: tensor([200.], device='cuda:0')
x2: tensor([100.], device='cuda:0')
y2: tensor([100.], device='cuda:0')
z2: tensor([100.], device='cuda:0')
s: tensor([8.5447], device='cuda:0')
s2: tensor([73.0115], device='cuda:0')
theta0: tensor([0.7071], device='cuda:0')
theta1: tensor([0.5103], device='cuda:0')
offset_r: tensor([-1.3485], device='cuda:0')
ra: tensor([12.7937], device='cuda:0')
za: tensor([-4.9727], device='cuda:0')
new_x: tensor([9.0465], device='cuda:0')
new_y: tensor([9.0465], de

In [4]:
print(f"dra_dr: {RT.dra_dr()}")
print(f"dra_dz: {RT.dra_dz()}")
print(f"dza_dr: {RT.dza_dr()}")
print(f"dza_dz: {RT.dza_dz()}")
print(f"dPa_dP: {RT.dPa_dP()}")


dra_dr: tensor([1.4827], device='cuda:0')
dra_dz: tensor([0.4694], device='cuda:0')
dza_dr: tensor([-0.0565], device='cuda:0')
dza_dz: tensor([0.7453], device='cuda:0')
dPa_dP: tensor([[[1.1937, 0.2890, 0.3319],
         [0.2890, 1.1937, 0.3319],
         [0.3319, 0.3319, 0.7453]]], device='cuda:0')


# 数値計算 VS 理論計算

In [5]:
delta = 0.01

In [6]:
means_p_dx = torch.Tensor([[x+delta, y, z]]).to(device)

RT_p_dx = RefractionTransform(
    means=means_p_dx,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)

means_m_dx = torch.Tensor([[x-delta, y, z]]).to(device)

RT_m_dx = RefractionTransform(
    means=means_m_dx,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)

means_app_m_dx = RT_m_dx.transform_to_appearance()
means_app = RT.transform_to_appearance()
means_app_p_dx = RT_p_dx.transform_to_appearance()

print(f"means_app: {means_app}")
print(f"means_app_dx: {means_app_m_dx}")
print(f"means_app_dx: {means_app_p_dx}")

dPa_dx = (means_app_p_dx - means_app_m_dx) / (2 * delta)
print(f"dxa_dx: {dPa_dx}")

means_app: tensor([[ 9.0465,  9.0465, -4.9727]], device='cuda:0')
means_app_dx: tensor([[ 9.0382,  9.0472, -4.9747]], device='cuda:0')
means_app_dx: tensor([[ 9.0548,  9.0458, -4.9707]], device='cuda:0')
dxa_dx: tensor([[ 0.8324, -0.0723,  0.1987]], device='cuda:0')


In [7]:
# dy
means_p_dy = torch.Tensor([[x, y+delta, z]]).to(device)
RT_p_dy = RefractionTransform(
    means=means_p_dy,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)

means_m_dy = torch.Tensor([[x, y-delta, z]]).to(device)
RT_m_dy = RefractionTransform(
    means=means_m_dy,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)

means_app_m_dy = RT_m_dy.transform_to_appearance()
means_app_dy = RT_p_dy.transform_to_appearance()
print(f"means_app: {means_app}")
print(f"means_app_dy: {means_app_dy}")
print(f"means_app_dy: {means_app_m_dy}")
dPa_dy = (means_app_dy - means_app_m_dy) / (2 * delta)
print(f"dPa_dy: {dPa_dy}")



means_app: tensor([[ 9.0465,  9.0465, -4.9727]], device='cuda:0')
means_app_dy: tensor([[ 9.0458,  9.0548, -4.9707]], device='cuda:0')
means_app_dy: tensor([[ 9.0472,  9.0382, -4.9747]], device='cuda:0')
dPa_dy: tensor([[-0.0724,  0.8323,  0.1987]], device='cuda:0')


In [8]:
# dz
means_p_dz = torch.Tensor([[x, y, z+delta]]).to(device)
RT_p_dz = RefractionTransform(
    means=means_p_dz,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)
means_m_dz = torch.Tensor([[x, y, z-delta]]).to(device)
RT_m_dz = RefractionTransform(
    means=means_m_dz,
    quats=None,
    cam_center=cam_center,
    n=n,
    plane=plane,
    num_newton_iters=num_newton_iters,
    newton_atol=newton_atol,
)
means_app_m_dz = RT_m_dz.transform_to_appearance()
means_app_dz = RT_p_dz.transform_to_appearance()
print(f"means_app: {means_app}")
print(f"means_app_dz: {means_app_dz}")
print(f"means_app_dz: {means_app_m_dz}")
dPa_dz = (means_app_dz - means_app_m_dz) / (2 * delta)
print(f"dPa_dz: {dPa_dz}")

means_app: tensor([[ 9.0465,  9.0465, -4.9727]], device='cuda:0')
means_app_dz: tensor([[ 9.0465,  9.0465, -4.9661]], device='cuda:0')
means_app_dz: tensor([[ 9.0465,  9.0465, -4.9792]], device='cuda:0')
dPa_dz: tensor([[    0.0004,     0.0003,     0.6545]], device='cuda:0')


In [17]:
dPa_dP_nd = torch.cat([dPa_dx, dPa_dy, dPa_dz], dim=0).transpose(0, 1)
print(f"dPa_dP_nd:\n {dPa_dP_nd}")
print(f"jacobian:\n {dPa_dP}")

dPa_dP_nd:
 tensor([[     0.8324,     -0.0724,      0.0004],
        [    -0.0723,      0.8323,      0.0003],
        [     0.1987,      0.1987,      0.6545]], device='cuda:0')
jacobian:
 tensor([[[1.1937, 0.2890, 0.3319],
         [0.2890, 1.1937, 0.3319],
         [0.3319, 0.3319, 0.7453]]], device='cuda:0')


In [18]:
dPa_dP_ = RT.dPa_dP_numerical()
print(f"dPa_dP_:\n {dPa_dP_}")

dPa_dP_:
 tensor([[[     0.8324,     -0.0724,      0.0004],
         [    -0.0723,      0.8323,      0.0003],
         [     0.1987,      0.1987,      0.6545]]], device='cuda:0')


/home/taiki/try-papers/gsplat/dev_refraction/refraction_transform.py:62: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.n = torch.tensor(n, dtype=torch.float32, device=device,
/home/taiki/try-papers/gsplat/dev_refraction/refraction_transform.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.plane = torch.tensor(plane, dtype=torch.float32, device=device,
